In [2]:
import os
os.listdir()



['.ipynb_checkpoints', 'data', 'data_quality_report.txt', 'etl_test.ipynb']

In [3]:
os.listdir('data')


['.ipynb_checkpoints',
 'Customers.csv',
 'customers_cleaned.csv',
 'products_cleaned.csv',
 'products_raw.csv',
 'sales_cleaned.csv',
 'sales_raw.csv']

In [4]:
import pandas as pd

import os
fnames = []
fpaths = []
for dirname, _, filenames in os.walk(r'C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data'):
    for filename in filenames:
        fnames.append(filename.split('.')[0])
        fpaths.append(os.path.join(dirname, filename))
        print(os.path.join(dirname, filename))

C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data\Customers.csv
C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data\customers_cleaned.csv
C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data\products_cleaned.csv
C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data\products_raw.csv
C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data\sales_cleaned.csv
C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data\sales_raw.csv


In [5]:
fpath = r'C:\Users\ishaan karwayun\OneDrive - Symbiosis International University\Desktop\BITSOM\studentID-fleximart-data-architecture\data'
customers_raw_df = pd.read_csv(os.path.join(fpath,'Customers.csv'),na_values = r'/N')
products_raw_df = pd.read_csv(os.path.join(fpath,'products_raw.csv'),na_values = r'/N')
sales_raw_df = pd.read_csv(os.path.join(fpath,'sales_raw.csv'),na_values = r'/N')

In [8]:
customers.head()

,customer_id,first_name,last_name,email,phone,city,registration_date
0,C001,Rahul,Sharma,rahul.sharma@gmail.com,9876543210,Bangalore,2023-01-15
1,C002,Priya,Patel,priya.patel@yahoo.com,+91-9988776655,Mumbai,2023-02-20
2,C003,Amit,Kumar,NaN,9765432109,Delhi,2023-03-10
3,C004,Sneha,Reddy,sneha.reddy@gmail.com,9123456789,Hyderabad,15/04/2023
4,C005,Vikram,Singh,vikram.singh@outlook.com,09988112233,Chennai,2023-05-22


In [7]:
#creating copies 
customers = customers_raw_df.copy()
products = products_raw_df.copy()
sales = sales_raw_df.copy()

In [9]:
#handling missing email values
missing_email_mask = customers['email'].isna() | (customers['email'].str.strip() == '')
customers.loc[missing_email_mask, 'email'] = (
    'unknown_' +
    customers.loc[missing_email_mask, 'customer_id'] +
    '@fleximart.com'
)


In [10]:
customers = customers.drop_duplicates()

In [11]:
customers['phone'].astype(str).unique()
customers['phone'].str.contains(r'[^0-9+]', regex=True).sum()


np.int64(2)

In [12]:
customers['phone'] = customers['phone'].astype(str).str.replace(r'\D', '', regex = True)
customers['phone'] = customers['phone'].str.lstrip('0')
customers['phone'] = customers['phone'].apply(lambda x: '+91' + x if len(x) == 10 else '+' + x if x.startswith('91') else x)


In [13]:
customers['registration_date'] = customers['registration_date'].str.replace('/', '-')
from datetime import datetime
def parse_mixed_date(x):
    for fmt in ("%d-%m-%Y", "%Y-%m-%d", "%m-%d-%Y"):
        try:
            return datetime.strptime(x, fmt)
        except:
            continue
    return x  # keep original if it doesn't match any
customers['registration_date'] = customers['registration_date'].apply(parse_mixed_date)


In [14]:
customers['city'] = customers['city'].str.strip()
customers['city'] = customers['city'].str.lower()

In [15]:
customers

,customer_id,first_name,last_name,email,phone,city,registration_date
0,C001,Rahul,Sharma,rahul.sharma@gmail.com,+919876543210,bangalore,2023-01-15
1,C002,Priya,Patel,priya.patel@yahoo.com,+919988776655,mumbai,2023-02-20
2,C003,Amit,Kumar,unknown_C003@fleximart.com,+919765432109,delhi,2023-03-10
3,C004,Sneha,Reddy,sneha.reddy@gmail.com,+919123456789,hyderabad,2023-04-15
4,C005,Vikram,Singh,vikram.singh@outlook.com,+919988112233,chennai,2023-05-22
5,C006,Anjali,Mehta,anjali.mehta@gmail.com,+919876543210,bangalore,2023-06-18
6,C007,Ravi,Verma,unknown_C007@fleximart.com,+919876501234,pune,2023-07-25
7,C008,Pooja,Iyer,pooja.iyer@gmail.com,+919123456780,bangalore,2023-08-15
8,C009,Karthik,Nair,karthik.nair@yahoo.com,+919988776644,kochi,2023-09-30
9,C010,Deepa,Gupta,deepa.gupta@gmail.com,+919871234567,delhi,2023-10-12


In [16]:
products['category'] = products['category'].str.strip()
products['category'] = products['category'].str.lower()

In [17]:
products['price'] = products['price'].fillna(0).astype(int)

In [18]:
category_price = products.groupby('category')['price'].agg('mean','median').reset_index().round(2)
category_price

,category,price
0,electronics,24277.00
1,fashion,2827.71
2,groceries,529.75


In [30]:
products['price'].replace(0, pd.NA)


In [19]:
#filling missing values with categorical means
products['price'] = products.groupby('category')['price'].transform(
    lambda x: x.where(x != 0, x.median())
).astype(int)

In [20]:
#filling missing stock qty with 0
products['stock_quantity'] = products['stock_quantity'].fillna(0).astype(int)

In [82]:
sales['transaction_date']=sales_raw_df['transaction_date']

In [24]:
sales['transaction_date'] = sales['transaction_date'].astype(str).str.replace('/','-')
sales['transaction_date'] = sales['transaction_date'].apply(parse_mixed_date)
sales['unit_price'] = sales['unit_price'].astype(int)

In [26]:
sales = sales.drop_duplicates()

In [27]:
report = f"""
CUSTOMERS
Records processed: {len(customers_raw_df)}
Duplicates removed: {customers_raw_df.duplicated().sum()}
Missing values handled: {customers_raw_df['email'].isna().sum()}
Records loaded: {len(customers)}

PRODUCTS
Records processed: {len(products_raw_df)}
Duplicates removed: {products_raw_df.duplicated().sum()}
Missing values handled: {products_raw_df.isna().sum()}
Records loaded: {len(products)}

SALES
Records processed: {len(sales_raw_df)}
Duplicates removed: {sales_raw_df.duplicated().sum()}
Missing values handled: {sales_raw_df.isna().sum()}
Records loaded: {len(sales)}
"""


In [28]:
with open("data_quality_report.txt", "w") as f:
    f.write(report)

In [108]:
products_raw_df.duplicated().sum()

np.int64(0)

In [111]:
customers.to_csv("data/customers_cleaned.csv", index=False)
products.to_csv("data/products_cleaned.csv", index=False)
sales.to_csv("data/sales_cleaned.csv", index=False)


In [29]:
pip install mysql-connector-python


Note: you may need to restart the kernel to use updated packages.


In [30]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="ONZIbonzi1234",  # replace with your MySQL password
    database="fleximart"
)
cursor = conn.cursor()


In [32]:
customers

,customer_id,first_name,last_name,email,phone,city,registration_date
0,C001,Rahul,Sharma,rahul.sharma@gmail.com,+919876543210,bangalore,2023-01-15
1,C002,Priya,Patel,priya.patel@yahoo.com,+919988776655,mumbai,2023-02-20
2,C003,Amit,Kumar,unknown_C003@fleximart.com,+919765432109,delhi,2023-03-10
3,C004,Sneha,Reddy,sneha.reddy@gmail.com,+919123456789,hyderabad,2023-04-15
4,C005,Vikram,Singh,vikram.singh@outlook.com,+919988112233,chennai,2023-05-22
5,C006,Anjali,Mehta,anjali.mehta@gmail.com,+919876543210,bangalore,2023-06-18
6,C007,Ravi,Verma,unknown_C007@fleximart.com,+919876501234,pune,2023-07-25
7,C008,Pooja,Iyer,pooja.iyer@gmail.com,+919123456780,bangalore,2023-08-15
8,C009,Karthik,Nair,karthik.nair@yahoo.com,+919988776644,kochi,2023-09-30
9,C010,Deepa,Gupta,deepa.gupta@gmail.com,+919871234567,delhi,2023-10-12


In [31]:
# Insert into MySQL
for _, row in customers.iterrows():
    try:
        cursor.execute("""
            INSERT INTO customers
            (first_name, last_name, email, phone, city, registration_date)
            VALUES (%s,%s,%s,%s,%s,%s)
        """, tuple(row[['first_name','last_name','email','phone','city','registration_date']]))
        report['customers']['loaded'] += 1
    except Exception as e:
        print("Customer insert failed:", e)

conn.commit()

Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not 'str'
Customer insert failed: string indices must be integers, not '

In [33]:
for _, row in products.iterrows():
    try:
        cursor.execute("""
            INSERT INTO products
            (product_name, category, price, stock_quantity)
            VALUES (%s,%s,%s,%s)
        """, tuple(row[['product_name','category','price','stock_quantity']]))
        report['products']['loaded'] += 1
    except Exception as e:
        print("products insert failed:", e)

conn.commit()

products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not 'str'
products insert failed: string indices must be integers, not '

In [58]:
for _, row in sales.iterrows():
    try:
        cursor.execute("""
            INSERT INTO orders
            (customer_id,order_date,total_amount, status)
            VALUES (%s,%s,%s,%s)
        """, tuple(row[['customer_sk','transaction_date','subtotal','status']]))
    except Exception as e:
        print("sales insert failed:", e)

conn.commit()

In [123]:
for _, row in sales.iterrows():
    try:
        cursor.execute("""
            INSERT INTO order_items
            (order_id,product_id,quantity,unit_price,subtotal)
            VALUES (%s,%s,%s,%s,%s)
        """, tuple(row[['order_sk','product_sk','quantity','unit_price','subtotal']]))
    except Exception as e:
        print("sales insert failed:", e)

conn.commit()

In [112]:
sales.head()

,transaction_id,customer_id,product_id,quantity,unit_price,transaction_date,status,customer_sk,subtotal,product_sk,order_sk
0,T001,C001,P001,1,45999,2024-01-15,Completed,1,45999,1,1
1,T002,C002,P004,2,2999,2024-01-16,Completed,2,5998,4,20
2,T003,C003,P007,1,52999,2024-01-15,Completed,3,52999,7,21
4,T005,C005,P009,3,650,2024-01-20,Completed,5,1950,9,23
5,T006,C006,P012,1,12999,2024-01-22,Completed,6,12999,12,24


In [56]:
sales['subtotal'] = sales['quantity'] * sales['unit_price']

C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_23796\3484039757.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['subtotal'] = sales['quantity'] * sales['unit_price']


In [41]:
customers

,first_name,last_name,email,phone,city,registration_date,customer_sk,raw_customer_id
0,Rahul,Sharma,rahul.sharma@gmail.com,+919876543210,bangalore,2023-01-15,1,C001
1,Priya,Patel,priya.patel@yahoo.com,+919988776655,mumbai,2023-02-20,2,C002
2,Amit,Kumar,unknown_C003@fleximart.com,+919765432109,delhi,2023-03-10,3,C003
3,Sneha,Reddy,sneha.reddy@gmail.com,+919123456789,hyderabad,2023-04-15,4,C004
4,Vikram,Singh,vikram.singh@outlook.com,+919988112233,chennai,2023-05-22,5,C005
5,Anjali,Mehta,anjali.mehta@gmail.com,+919876543210,bangalore,2023-06-18,6,C006
6,Ravi,Verma,unknown_C007@fleximart.com,+919876501234,pune,2023-07-25,7,C007
7,Pooja,Iyer,pooja.iyer@gmail.com,+919123456780,bangalore,2023-08-15,8,C008
8,Karthik,Nair,karthik.nair@yahoo.com,+919988776644,kochi,2023-09-30,9,C009
9,Deepa,Gupta,deepa.gupta@gmail.com,+919871234567,delhi,2023-10-12,10,C010


In [35]:
# calling sk customer_id from sql
cursor.execute("""
SELECT customer_id, email, first_name, last_name
FROM customers
""")
rows = cursor.fetchall()


In [36]:
sql_customers = pd.DataFrame(rows, columns=['customer_id', 'email', 'first_name', 'last_name']
)


In [38]:
customers['customer_sk'] = sql_customers['customer_id'].values


In [40]:
customers['raw_customer_id'] = customers['customer_id']
customers = customers.drop('customer_id',axis = 1)

In [45]:
#mapping old customer_id 
customer_map = dict(
    zip(customers['raw_customer_id'], customers['customer_sk'])
)

In [46]:
sales['customer_sk'] = sales['customer_id'].map(customer_map)


C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_23796\4040543486.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['customer_sk'] = sales['customer_id'].map(customer_map)


In [51]:
sales = sales.dropna(subset =['customer_id'])

In [69]:
sales['customer_sk'] = sales['customer_sk'].astype(int)

C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_45816\3654214341.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['customer_sk'] = sales['customer_sk'].astype(int)


In [54]:
#converting the column to be inserted
sales['customer_sk'] = sales['customer_sk'].astype(int)
sales

,transaction_id,customer_id,product_id,quantity,unit_price,transaction_date,status,customer_sk
0,T001,C001,P001,1,45999,2024-01-15,Completed,1
1,T002,C002,P004,2,2999,2024-01-16,Completed,2
2,T003,C003,P007,1,52999,2024-01-15,Completed,3
4,T005,C005,P009,3,650,2024-01-20,Completed,5
5,T006,C006,P012,1,12999,2024-01-22,Completed,6
6,T007,C007,P005,2,1999,2024-01-23,Completed,7
7,T008,C008,NaN,1,1299,2024-01-25,Completed,8
8,T009,C009,P011,1,4599,2024-01-28,Cancelled,9
9,T010,C010,P006,5,899,2024-02-01,Completed,10
11,T011,C011,P014,1,69999,2024-02-02,Completed,11


In [96]:
cursor.execute("""
SELECT order_id, customer_id, order_date, status
FROM orders
""")
rows = cursor.fetchall()

In [97]:
sql_orders =  pd.DataFrame(rows, columns=['order_id', 'customer_id', 'order_date', 'state'])

In [98]:
sales['order_id'] = sql_orders['order_id']

C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_45816\765141022.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['order_id'] = sql_orders['order_id']


In [104]:
sales = sales.dropna(subset=['product_id'])


In [107]:
sales['product_id'] = sales_raw_df['product_id']

C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_45816\3917478489.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['product_id'] = sales_raw_df['product_id']


In [63]:
sales

,transaction_id,customer_id,product_id,quantity,unit_price,transaction_date,status,customer_sk,subtotal
0,T001,C001,P001,1,45999,2024-01-15,Completed,1,45999
1,T002,C002,P004,2,2999,2024-01-16,Completed,2,5998
2,T003,C003,P007,1,52999,2024-01-15,Completed,3,52999
4,T005,C005,P009,3,650,2024-01-20,Completed,5,1950
5,T006,C006,P012,1,12999,2024-01-22,Completed,6,12999
6,T007,C007,P005,2,1999,2024-01-23,Completed,7,3998
7,T008,C008,NaN,1,1299,2024-01-25,Completed,8,1299
8,T009,C009,P011,1,4599,2024-01-28,Cancelled,9,4599
9,T010,C010,P006,5,899,2024-02-01,Completed,10,4495
11,T011,C011,P014,1,69999,2024-02-02,Completed,11,69999


In [64]:
cursor.execute("""
SELECT *
FROM products
""")
product_rows = cursor.fetchall()

In [70]:
sql_products = pd.DataFrame(product_rows, columns=['product_id', 'product_name', 'category ', 'price ','stock_quantity'])


In [74]:
products['product_sk'] = sql_products['product_id']

In [78]:
product_map = dict(zip(products['product_id'],
                        products['product_sk']))

In [87]:
#maping sales product id 
sales['product_sk'] = sales['product_id'].map(product_map)


C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_23796\3221493123.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['product_sk'] = sales['product_id'].map(product_map)


In [90]:
sales = sales.dropna(subset = ['product_sk'])

In [91]:
sales['product_sk'] = sales['product_sk'].astype(int)

C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_23796\598957504.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['product_sk'] = sales['product_sk'].astype(int)


In [93]:
#calling order sk from sql
cursor.execute("""
select * from orders """)
order_rows = cursor.fetchall()

In [94]:
sql_orders = pd.DataFrame(order_rows,columns = ['order_id','customer_id','order_date','total_amount','status'])

In [97]:
sales['order_sk'] = sql_orders['order_id']

C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_23796\2768416287.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['order_sk'] = sql_orders['order_id']


In [105]:
order_map = dict(zip(sql_orders['customer_id'], sql_orders['order_id']))

In [108]:
sales['order_sk'] = sales['customer_sk'].map(order_map)

C:\Users\ishaan karwayun\AppData\Local\Temp\ipykernel_23796\739977595.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales['order_sk'] = sales['customer_sk'].map(order_map)


In [110]:
order_map

{1: 1,
 2: 20,
 3: 21,
 5: 23,
 6: 24,
 7: 25,
 8: 26,
 9: 27,
 10: 9,
 11: 28,
 12: 29,
 13: 30,
 14: 31,
 15: 32,
 17: 34,
 18: 35,
 19: 36,
 20: 37,
 21: 19,
 4: 22,
 16: 33}

In [118]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35 entries, 0 to 40
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    35 non-null     object        
 1   customer_id       35 non-null     object        
 2   product_id        35 non-null     object        
 3   quantity          35 non-null     int64         
 4   unit_price        35 non-null     int64         
 5   transaction_date  35 non-null     datetime64[ns]
 6   status            35 non-null     object        
 7   customer_sk       35 non-null     int64         
 8   subtotal          35 non-null     int64         
 9   product_sk        35 non-null     int64         
 10  order_sk          35 non-null     int64         
dtypes: datetime64[ns](1), int64(6), object(4)
memory usage: 3.3+ KB
